In [ ]:
from pathlib import Path
from typing import Sequence
import time

import chipwhisperer as cw


# Current LESS firmware uses the official toy parameters.
Q = 127
N = 16
K = 8
N_PAD = 16

FQ_ELEM_BYTES = 1
POSITION_T_BYTES = 2

GENERATOR_BYTES = K * N_PAD * FQ_ELEM_BYTES
COEFFICIENT_BYTES = N * FQ_ELEM_BYTES
PERMUTATION_BYTES = N * POSITION_T_BYTES

INPUT_BYTES = (
    GENERATOR_BYTES
    + COEFFICIENT_BYTES
    + PERMUTATION_BYTES
)

OUTPUT_BYTES = GENERATOR_BYTES

# Must match USB_PACKET_DATABUFFER_SIZE in main.c.
CHUNK_SIZE = 160

SUBROUTINE_GENERATOR_MONOMIAL_MUL = 0x03
BAUDRATE = 230400

FIRMWARE_DIRECTORY = Path.cwd()
HEX_BINARY = (
    FIRMWARE_DIRECTORY
    / "less_generator_monomial_mul-PINATA.hex"
)

print("Input bytes :", INPUT_BYTES)
print("Output bytes:", OUTPUT_BYTES)
print("HEX file    :", HEX_BINARY)

In [ ]:
def serialize_generator_monomial_mul(
    generator_values: Sequence[int],
    coefficients: Sequence[int],
    permutation: Sequence[int],
) -> bytes:
    """
    Serialize the input exactly as expected by less_monomial_mul.c.

    Layout:
        1. K * N_PAD generator values in row-major order
        2. N monomial coefficients
        3. N permutation entries as little-endian uint16 values
    """

    if len(generator_values) != K * N_PAD:
        raise ValueError(
            f"Expected {K * N_PAD} generator values, "
            f"got {len(generator_values)}"
        )

    if len(coefficients) != N:
        raise ValueError(
            f"Expected {N} coefficients, "
            f"got {len(coefficients)}"
        )

    if len(permutation) != N:
        raise ValueError(
            f"Expected {N} permutation entries, "
            f"got {len(permutation)}"
        )

    if any(value < 0 or value >= Q for value in generator_values):
        raise ValueError(
            f"Generator values must be in [0, {Q - 1}]"
        )

    if any(value <= 0 or value >= Q for value in coefficients):
        raise ValueError(
            f"Coefficients must be in [1, {Q - 1}]"
        )

    if sorted(permutation) != list(range(N)):
        raise ValueError(
            f"Permutation must contain every index from 0 to {N - 1}"
        )

    payload = bytearray()

    # FQ_ELEM is uint8_t.
    payload.extend(generator_values)
    payload.extend(coefficients)

    # POSITION_T is uint16_t; firmware expects little-endian order.
    for position in permutation:
        payload.extend(int(position).to_bytes(
            POSITION_T_BYTES,
            byteorder="little",
            signed=False,
        ))

    if len(payload) != INPUT_BYTES:
        raise RuntimeError(
            f"Serialized payload has {len(payload)} bytes, "
            f"expected {INPUT_BYTES}"
        )

    return bytes(payload)

In [ ]:
def connect_and_flash():
    """
    Connect to the ChipWhisperer Nano, configure the 15 MHz clock,
    flash the LESS firmware, and create a SimpleSerial2 target.
    """

    if not HEX_BINARY.is_file():
        raise FileNotFoundError(
            f"Firmware image not found: {HEX_BINARY}"
        )

    scope = cw.scope()
    scope.default_setup()

    scope.io.clkout = 15e6
    print("Clock output:", scope.io.clkout)

    time.sleep(0.5)

    cw.program_target(
        scope,
        cw.programmers.STM32FProgrammer,
        str(HEX_BINARY),
        baud=BAUDRATE,
    )

    target = cw.target(
        scope,
        cw.targets.SimpleSerial2,
    )

    target.baud = BAUDRATE
    target.flush()

    return scope, target


def upload_generator_monomial_mul_input(
    target,
    payload: bytes,
    verify_readback: bool = True,
) -> None:
    """
    Upload the complete 176-byte input in chunks.

    The subcommand's lower seven bits contain the chunk index.
    Bit 7 requests a readback from the firmware.
    """

    if len(payload) != INPUT_BYTES:
        raise ValueError(
            f"Expected {INPUT_BYTES} input bytes, "
            f"got {len(payload)}"
        )

    for chunk_index, offset in enumerate(
        range(0, len(payload), CHUNK_SIZE)
    ):
        chunk = payload[offset:offset + CHUNK_SIZE]

        subcommand = chunk_index

        if verify_readback:
            subcommand |= 0x80

        target.send_cmd(
            "s",
            subcommand,
            bytearray(chunk),
        )

        if verify_readback:
            readback = target.simpleserial_read(
                "r",
                len(chunk),
                timeout=1000,
                ack=True,
            )

            if readback is None:
                raise TimeoutError(
                    f"No readback received for chunk {chunk_index}"
                )

            if bytes(readback) != chunk:
                raise RuntimeError(
                    f"Readback mismatch for chunk {chunk_index}"
                )
        else:
            acknowledgement = target.simpleserial_wait_ack(
                timeout=1000
            )

            if acknowledgement is None:
                raise TimeoutError(
                    f"No acknowledgement for chunk {chunk_index}"
                )


def generator_monomial_mul_board(
    target,
    generator_values: Sequence[int],
    coefficients: Sequence[int],
    permutation: Sequence[int],
) -> list[int]:
    """
    Execute generator_monomial_mul on the Pinata and return the
    flattened K x N_PAD result matrix.
    """

    payload = serialize_generator_monomial_mul(
        generator_values,
        coefficients,
        permutation,
    )

    upload_generator_monomial_mul_input(
        target,
        payload,
        verify_readback=True,
    )

    # Command 'b', subcommand 0x03 executes generator_monomial_mul.
    target.send_cmd(
        "b",
        SUBROUTINE_GENERATOR_MONOMIAL_MUL,
        bytearray(),
    )

    response = target.simpleserial_read(
        "r",
        OUTPUT_BYTES,
        timeout=2000,
        ack=True,
    )

    if response is None:
        raise TimeoutError(
            "No result received from generator_monomial_mul"
        )

    if len(response) != OUTPUT_BYTES:
        raise RuntimeError(
            f"Expected {OUTPUT_BYTES} result bytes, "
            f"received {len(response)}"
        )

    return list(response)

In [ ]:
def generator_monomial_mul_python(
    generator_values: Sequence[int],
    coefficients: Sequence[int],
    permutation: Sequence[int],
) -> list[int]:
    """
    Independent Python calculation for the current toy parameters.
    """

    result = [0] * (K * N_PAD)

    for source_column in range(N):
        destination_column = permutation[source_column]
        coefficient = coefficients[source_column]

        for row in range(K):
            source_index = row * N_PAD + source_column
            destination_index = (
                row * N_PAD + destination_column
            )

            result[destination_index] = (
                generator_values[source_index]
                * coefficient
            ) % Q

    return result


generator = [
    (row * N_PAD + column) % Q
    for row in range(K)
    for column in range(N_PAD)
]

coefficients = [
    (index % (Q - 1)) + 1
    for index in range(N)
]

permutation = list(reversed(range(N)))

expected_result = generator_monomial_mul_python(
    generator,
    coefficients,
    permutation,
)

scope = None
target = None

try:
    scope, target = connect_and_flash()

    board_result = generator_monomial_mul_board(
        target,
        generator,
        coefficients,
        permutation,
    )

    print("Expected first row:", expected_result[:N_PAD])
    print("Board first row   :", board_result[:N_PAD])

    if board_result != expected_result:
        raise AssertionError(
            "Pinata result does not match the Python result"
        )

    print("OK: Pinata and Python results match.")

finally:
    if target is not None:
        target.dis()

    if scope is not None:
        scope.dis()